# Qwen3-4B / SynthID: exact 96-generation replication
Native, self-contained Colab cells; no external experiment imports, CLI worker, or chat template.
**This is raw English story continuation, not Qwen's standard assistant mode.** No `apply_chat_template`, reasoning prompt, `<think>` prefix, or `enable_thinking` switch is used. It preserves the original distilgpt2 experiment's literal prompts, seeds, splits, sampling, perturbations, scoring masks and calibration rule; changing checkpoint, tokenizer, CUDA and bfloat16 is NOT a single-variable CPU/GPU comparison. The original CPU distilgpt2 experiment was real model inference.

Run all on an already-authorized CUDA Colab runtime with HF_TOKEN injected into the kernel environment by the host. This notebook does not allocate, authenticate Colab, or stop a VM. Host lifecycle must enforce the approved cumulative 60-minute budget, including installation/download/verification.

64 and 128 new tokens × (24 plain calibration + 12 plain evaluation + 12 watermarked evaluation) = **96 generations**. Four paired token-ID conditions × correct/wrong keys = **768 score rows**, **16 separately calibrated threshold groups**. Keys are public reproduction keys, not a production detector. Mean g-value is not a Bayesian posterior. No quality, semantic paraphrase, or product detection claim.

Outputs: `/content/synthid-qwen-output/{generations.jsonl,real.json,summary.png,provenance.json,audit.json,results.zip}`. JSONL flushes after every generation. Existing artifacts are never silently overwritten; use a new OUTPUT directory to rerun.

Optional CPU saved-result replay: set MODE='replay', REPLAY_SOURCE to a completed output directory, and OUTPUT to a new directory. No model/tokenizer download or generation occurs; original GPU timings are historical. Replay checks stored IDs, edits, g-values, masks, thresholds and decisions, but cannot independently validate decoded text without the pinned tokenizer.


In [ ]:
import os, sys, time
from pathlib import Path
MODE = 'generate'  # 'generate' or 'replay'; replay requires saved real.json + generations.jsonl
REPLAY_SOURCE = Path('/content/synthid-qwen-output')
OUTPUT = Path('/content/synthid-qwen-output') if MODE == 'generate' else Path('/content/synthid-qwen-replay')
assert MODE in ('generate', 'replay')
# Must precede torch/CUDA and transformers imports. Do not suppress warnings.
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
NOTEBOOK_STARTED = time.perf_counter()


In [ ]:
# Isolated system-site-packages venv, but execute native cells in THIS kernel.
# Retain the base torch/CUDA wheel and base NumPy ABI rather than downloading torch.
import importlib.metadata as im, json, site, subprocess, venv
BASE_TORCH = im.version('torch')
BASE_NUMPY = im.version('numpy')
BASE_MATPLOTLIB = im.version('matplotlib')
assert (1, 26) <= tuple(map(int, BASE_NUMPY.split('.')[:2])) < (3, 0), 'Unsupported base NumPy'
ENV = Path('/content/synthid-qwen-env')
# Colab omits ensurepip: use the base pip to manage a pip-less isolated venv.
venv.EnvBuilder(system_site_packages=True, with_pip=False).create(ENV)
constraint = ENV / 'base-constraints.txt'
constraint.write_text(f'torch=={BASE_TORCH}\nnumpy=={BASE_NUMPY}\n')
requirements = ['transformers==4.57.1', 'tokenizers==0.22.1',
                'huggingface-hub==0.36.0', 'safetensors==0.6.2',
                f'numpy=={BASE_NUMPY}', f'matplotlib=={BASE_MATPLOTLIB}']
install_env = dict(os.environ)
install_env.pop('UV_SYSTEM_PYTHON', None)
t0 = time.perf_counter()
install_env.pop('HF_TOKEN', None)  # package installation does not need model credentials
subprocess.run([sys.executable, '-m', 'pip', '--python', str(ENV/'bin/python'), 'install', '--disable-pip-version-check',
                '--constraint', str(constraint), *requirements], check=True, env=install_env)
site_dir = ENV / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
site.addsitedir(str(site_dir))
sys.path.remove(str(site_dir)); sys.path.insert(0, str(site_dir))
INSTALL_SECONDS = time.perf_counter() - t0
# Detect unsafe reuse of previously imported packages instead of hiding ABI/API mismatches.
for package, expected in [('transformers','4.57.1'), ('matplotlib',BASE_MATPLOTLIB), ('numpy',BASE_NUMPY)]:
    loaded = sys.modules.get(package)
    if loaded is not None and getattr(loaded, '__version__', expected) != expected:
        raise RuntimeError(f'{package} already loaded with another version: restart kernel and Run all')
assert im.version('torch') == BASE_TORCH
print(json.dumps({'base_torch_retained': BASE_TORCH, 'base_numpy_retained': BASE_NUMPY,
                  'installation_seconds': INSTALL_SECONDS}))


{"base_torch_retained": "2.11.0+cu130", "base_numpy_retained": "2.4.6", "installation_seconds": 11.57539908199999}


In [ ]:
import collections, hashlib, math, platform, random, zipfile
from datetime import datetime, timezone
from typing import TypedDict
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, GenerationConfig,
                          SynthIDTextWatermarkingConfig, SynthIDTextWatermarkLogitsProcessor, set_seed)
assert im.version('transformers') == '4.57.1'
assert torch.__version__ == BASE_TORCH
MODEL_ID = 'Qwen/Qwen3-4B'
REVISION = '1cfa9a7208912126459214e8b04321603b3df60c'
LENGTHS = [64, 128]
CALIBRATION_COUNT, EVALUATION_COUNT = 24, 12
KEYS = [654, 400, 836, 123, 340, 443, 597, 160]
WRONG_KEYS = [key + 10000 for key in KEYS]
CONDITIONS = ['clean', 'truncate_half', 'delete_10pct', 'replace_10pct']
torch.set_num_threads(4)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    assert torch.get_num_interop_threads() == 1, 'Restart kernel to set interop threads'
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
OUTPUT.mkdir(parents=True, exist_ok=True)
assert not any((OUTPUT/name).exists() for name in ('real.json','generations.jsonl','results.zip')), 'Use a fresh OUTPUT directory'
def save_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False)+'\n')


In [ ]:
class SamplePlan(TypedDict):
    prompt: str
    seed: int
    sample_index: int

CAL_TOPICS = ['a coastal lighthouse', 'a mountain observatory', 'a quiet library', 'an ancient bridge',
              'a garden in spring', 'a railway station', 'an ocean expedition', 'a pottery workshop',
              'a solar panel', 'a rainy afternoon', 'a desert caravan', 'a village bakery',
              'a river ecosystem', 'a museum collection', 'a forest trail', 'a night sky',
              'a bicycle journey', 'a wind farm', 'a small theater', 'a winter cabin',
              'a coral reef', 'an apple orchard', 'a clock repair shop', 'a ferry crossing']
EVAL_TOPICS = ['a community radio show', 'a robot learning chess', 'a city rooftop', 'a glass sculpture',
               'a migratory bird', 'a jazz rehearsal', 'a forgotten recipe', 'a volcanic island',
               'a newspaper archive', 'a public swimming pool', 'a music classroom', 'a satellite launch']

def sample_plan(length: int, split: str, count: int) -> list[SamplePlan]:
    if split not in ('calibration', 'evaluation') or count < 1:
        raise ValueError('invalid split/count')
    topics = CAL_TOPICS if split == 'calibration' else EVAL_TOPICS
    if count > len(topics):
        raise ValueError(f'maximum {len(topics)} distinct prompts for {split}')
    base = (100000 if split == 'calibration' else 200000) + length * 100
    return [SamplePlan(prompt=f'Write a short English story about {topic}. The story begins:',
                 seed=base + i, sample_index=i) for i, topic in enumerate(topics[:count])]

def perturb(tokens, condition, seed, vocab_size):
    out = list(tokens)
    rng = random.Random(seed)
    edits = []
    if condition == 'truncate_half':
        return out[:len(out)//2], list(range(len(out)//2, len(out)))
    if condition not in CONDITIONS:
        raise ValueError(condition)
    if condition in ('delete_10pct', 'replace_10pct'):
        edits = sorted(rng.sample(range(len(out)), len(out)//10))
        if condition == 'delete_10pct':
            edited = set(edits)
            out = [t for i, t in enumerate(out) if i not in edited]
        else:
            for i in edits:
                replacement = rng.randrange(vocab_size - 2)
                out[i] = replacement + (replacement >= out[i])
    return out, edits

def group_key(row):
    return f"{row['source_length']}:{row['condition']}:{row['key_condition']}"

def calibrate(rows):
    if not rows or any(r['split'] != 'calibration' or r['origin'] != 'plain' for r in rows):
        raise ValueError('thresholds require nonempty calibration plain rows only')
    if any(r['score'] is None or not math.isfinite(r['score']) for r in rows):
        raise ValueError('cannot calibrate empty or nonfinite scores')
    return max(r['score'] for r in rows) + 1e-12

def real_score(tokens, processor, eos_token_id):
    # Mirrors official detector masks, but uses a simple mean, NOT a trained posterior.
    if len(tokens) < processor.ngram_len:
        return {'score': None, 'effective_tokens': 0, 'valid_mask': [], 'g_values': []}
    import torch
    ids = torch.tensor([tokens], dtype=torch.long)
    g = processor.compute_g_values(ids)
    mask = processor.compute_context_repetition_mask(ids)
    mask &= processor.compute_eos_token_mask(ids, eos_token_id)[:, processor.ngram_len - 1:]
    count = int(mask.sum().item())
    return {'score': float(g[mask].float().mean().item()) if count else None,
            'effective_tokens': count, 'valid_mask': mask[0].tolist(), 'g_values': g[0].tolist()}

def summarize(rows):
    groups = {}
    for row in rows:
        groups.setdefault(group_key(row), []).append(row)
    thresholds, summary = {}, []
    for key, group in sorted(groups.items()):
        cal = [r for r in group if r['split'] == 'calibration']
        threshold = calibrate(cal)
        thresholds[key] = threshold
        info = {k: group[0][k] for k in ['source_length', 'condition', 'key_condition']}
        info.update(threshold=threshold, calibration_count=len(cal),
                    calibration_detected=sum(r['score'] > threshold for r in cal))
        for origin in ['plain', 'watermarked']:
            ev = [r for r in group if r['split']=='evaluation' and r['origin']==origin]
            info[f'{origin}_count'] = len(ev)
            info[f'{origin}_detected'] = sum(r['score'] is not None and r['score'] > threshold for r in ev)
            info[f'{origin}_unscorable'] = sum(r['score'] is None for r in ev)
            info[f'{origin}_mean_effective_tokens'] = sum(r['effective_tokens'] for r in ev)/len(ev) if ev else None
        summary.append(info)
    return summary, thresholds


## Portable official watermark table (important CUDA/CPU detail)
Transformers 4.57.1 constructs its random sampling table using a generator on the processor device. CPU and CUDA RNG algorithms can yield different tables for seed 0. Blindly generating with a CUDA table and scoring with a CPU table would test mismatched watermarks, despite identical keys.

The subclass below preserves the **original CPU-seeded sampling table** and copies the official processor's table and keys to CUDA before generation. All generation state and logits processing run on CUDA. Official generation ordering remains unchanged: watermarking is installed by `watermarking_config` after top-k, not manually inserted ahead of sampling warpers. CPU scoring uses the same canonical table and the **verbatim original `real_score`**, including its original EOS and repeated-context masks. This explicit table portability adaptation is recorded in parameters/provenance; it does not claim arbitrary GPU/CPU bitwise model reproducibility.


In [ ]:
class PortableSynthIDConfig(SynthIDTextWatermarkingConfig):
    def construct_processor(self, vocab_size, device):
        processor = super().construct_processor(vocab_size, torch.device('cpu'))
        processor.device = torch.device(device)
        processor.keys = processor.keys.to(processor.device)
        processor.sampling_table = processor.sampling_table.to(processor.device)
        return processor

def watermark_config(wrong=False):
    return PortableSynthIDConfig(ngram_len=5, keys=WRONG_KEYS if wrong else KEYS,
        sampling_table_seed=0, sampling_table_size=65536, context_history_size=1024,
        skip_first_ngram_calls=True)

def make_processor(wrong=False, device='cpu'):
    return watermark_config(wrong).construct_processor(0, torch.device(device))

processors = {'correct': make_processor(), 'wrong': make_processor(True)}
TABLE_SHA256 = hashlib.sha256(processors['correct'].sampling_table.numpy().tobytes()).hexdigest()
for length in LENGTHS:
    cal = sample_plan(length, 'calibration', CALIBRATION_COUNT)
    ev = sample_plan(length, 'evaluation', EVALUATION_COUNT)
    assert not ({r['prompt'] for r in cal} & {r['prompt'] for r in ev})
    assert not ({r['seed'] for r in cal} & {r['seed'] for r in ev})


In [ ]:
load_seconds = 0.0
if MODE == 'generate':
    assert torch.cuda.is_available(), 'A CUDA GPU is required for fresh generation'
    assert torch.cuda.is_bf16_supported(), 'This path requires CUDA bfloat16 support'
    assert 'L4' in torch.cuda.get_device_name(0), 'Only the authorized L4 GPU is permitted'
    assert os.environ.get('HF_TOKEN'), 'Host must inject HF_TOKEN into kernel environment (never print it)'
    # Validate identity without echoing credentials or server exceptions.
    from huggingface_hub import HfApi
    try:
        HfApi(token=os.environ['HF_TOKEN']).whoami()
    except Exception:
        raise RuntimeError('HF identity validation failed; check host token injection') from None
    t0 = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION,
        token=os.environ['HF_TOKEN'], trust_remote_code=False)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION,
        token=os.environ['HF_TOKEN'], trust_remote_code=False, use_safetensors=True,
        dtype=torch.bfloat16, attn_implementation='eager').to('cuda')
    model.eval()
    torch.cuda.synchronize()
    load_seconds = time.perf_counter()-t0
    assert next(model.parameters()).device.type == 'cuda'
    assert next(model.parameters()).dtype == torch.bfloat16
    EOS = tokenizer.eos_token_id
    assert isinstance(EOS, int), 'Use scalar tokenizer EOS to preserve original scorer contract'
    VOCAB_SIZE = tokenizer.vocab_size
    probe = make_processor(device='cuda')
    assert probe.device.type == probe.keys.device.type == probe.sampling_table.device.type == 'cuda'
    assert torch.equal(probe.sampling_table.cpu(), processors['correct'].sampling_table)
    # Table + g-value parity before any model generation (not an extra generation).
    probe_ids = torch.tensor([[10,11,12,13,14,15,16,17]], dtype=torch.long)
    assert torch.equal(probe.compute_g_values(probe_ids.cuda()).cpu(),
                       processors['correct'].compute_g_values(probe_ids))
    del probe
    # Do not inherit Qwen's multiple EOS IDs or checkpoint-specific sampling defaults.
    generation_defaults = dict(do_sample=True, temperature=1.0, top_k=40, top_p=1.0,
        num_beams=1, use_cache=True, pad_token_id=EOS, eos_token_id=EOS,
        bos_token_id=tokenizer.bos_token_id)
    params = dict(model_id=MODEL_ID, model_revision=REVISION, lengths=LENGTHS,
        calibration_count=CALIBRATION_COUNT, evaluation_count_per_origin=EVALUATION_COUNT,
        watermark_config=watermark_config().to_dict(), wrong_keys=WRONG_KEYS, threads=4,
        tokenizer_vocab_size=VOCAB_SIZE, tokenizer_eos_token_id=EOS,
        checkpoint_eos_token_ids=model.generation_config.eos_token_id,
        detector='Mean g-values over official EOS/repeated-context mask; NOT Bayesian probability',
        threshold_method='maximum plain calibration score + 1e-12, separately by source_length/condition/key',
        generation=dict(**generation_defaults, min_new_tokens='source_length', max_new_tokens='source_length', use_model_defaults=False),
        perturbation='token-ID edits only: floor(length/10) deletions/replacements; truncate to first half',
        prompt_policy='Original raw English story continuation; no chat template, no think prefix or reasoning switch',
        scoring_input='generated token IDs only; original real_score verbatim; retokenized diagnostics have no decisions',
        generation_device='cuda', scoring_device='cpu', dtype='bfloat16', attention='eager',
        sampling_table_policy='original CPU-seeded table copied to CUDA; official post-warper insertion',
        sampling_table_sha256=TABLE_SHA256)
    metadata = dict(timestamp_utc=datetime.now(timezone.utc).isoformat(), python=platform.python_version(),
        platform=platform.platform(), machine=platform.machine(), cpu=platform.processor(), cpu_count=os.cpu_count(),
        torch_threads=torch.get_num_threads(), torch_interop_threads=torch.get_num_interop_threads(),
        torch_cuda=torch.version.cuda, gpu_name=torch.cuda.get_device_name(0),
        gpu_properties=str(torch.cuda.get_device_properties(0)),
        deterministic_algorithms=torch.are_deterministic_algorithms_enabled(),
        cublas_workspace_config=os.environ['CUBLAS_WORKSPACE_CONFIG'], tf32=False,
        versions={p: im.version(p) for p in ['torch','transformers','tokenizers','huggingface-hub','safetensors','numpy','matplotlib']})
    save_json(OUTPUT/'environment.json', dict(parameters=params, environment=metadata))
    print(json.dumps({'model':MODEL_ID, 'revision':REVISION, 'gpu':metadata['gpu_name'],
                      'torch':torch.__version__, 'cuda':torch.version.cuda, 'dtype':'bfloat16'}))
else:
    saved = json.loads((REPLAY_SOURCE/'real.json').read_text())
    assert saved['parameters']['model_id'] == MODEL_ID and saved['parameters']['model_revision'] == REVISION
    params, metadata = saved['parameters'], saved['environment']
    assert params['sampling_table_sha256'] == TABLE_SHA256
    EOS, VOCAB_SIZE = params['tokenizer_eos_token_id'], params['tokenizer_vocab_size']


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

{"model": "Qwen/Qwen3-4B", "revision": "1cfa9a7208912126459214e8b04321603b3df60c", "gpu": "NVIDIA L4", "torch": "2.11.0+cu130", "cuda": "13.0", "dtype": "bfloat16"}


In [ ]:
# One independently seeded generate call per sequence. No hidden pilot generations.
rows, generations = [], []
generation_start = time.perf_counter()
if MODE == 'generate':
    torch.cuda.reset_peak_memory_stats()
    with (OUTPUT/'generations.jsonl').open('w') as log:
        for length in LENGTHS:
            for split, count, origins in [('calibration', CALIBRATION_COUNT, ['plain']),
                                          ('evaluation', EVALUATION_COUNT, ['plain','watermarked'])]:
                for plan in sample_plan(length, split, count):
                    # Verbatim raw input: no chat template, no think prefix/suffix.
                    inputs = tokenizer(plan['prompt'], return_tensors='pt').to('cuda')
                    for origin in origins:
                        set_seed(plan['seed'])
                        config = GenerationConfig(**generation_defaults, min_new_tokens=length,
                                                  max_new_tokens=length)
                        torch.cuda.synchronize()
                        t0 = time.perf_counter()
                        with torch.inference_mode():
                            sequence = model.generate(**inputs, generation_config=config, use_model_defaults=False,
                                watermarking_config=watermark_config() if origin=='watermarked' else None)
                        torch.cuda.synchronize()
                        elapsed = time.perf_counter()-t0
                        ids = sequence[0, inputs['input_ids'].shape[1]:].tolist()
                        assert len(ids) == length, (length, len(ids))
                        source_id = f'{length}-{split}-{origin}-{plan["sample_index"]:02d}'
                        generation = dict(source_id=source_id, source_length=length, split=split, origin=origin,
                            **plan, model_id=MODEL_ID, model_revision=REVISION, watermark_keys=KEYS,
                            prompt_token_ids=inputs['input_ids'][0].tolist(), token_ids=ids,
                            text=tokenizer.decode(ids, clean_up_tokenization_spaces=False),
                            generation_seconds=elapsed)
                        generations.append(generation)
                        log.write(json.dumps(generation, allow_nan=False)+'\n'); log.flush()
                        # Recompute contexts/masks independently after each perturbation.
                        for condition in CONDITIONS:
                            edit_seed = plan['seed'] + 900000
                            edited, edits = perturb(ids, condition, edit_seed, VOCAB_SIZE)
                            text = tokenizer.decode(edited, clean_up_tokenization_spaces=False)
                            reencoded = tokenizer.encode(text, add_special_tokens=False)
                            for key_name, processor in processors.items():
                                row = dict(source_id=source_id, source_length=length, split=split, origin=origin,
                                    **plan, condition=condition, perturbation_seed=edit_seed,
                                    edited_original_positions=edits, key_condition=key_name,
                                    token_ids=edited, text=text, actual_token_count=len(edited),
                                    roundtrip_equal=(edited==reencoded), retokenized_token_ids=reencoded,
                                    generation_seconds=elapsed, **real_score(edited, processor, EOS))
                                rt = real_score(reencoded, processor, EOS)
                                row['retokenized_score'] = rt['score']
                                row['retokenized_effective_tokens'] = rt['effective_tokens']
                                rows.append(row)
                        print(f'{source_id}: {length} new tokens in {elapsed:.3f}s', flush=True)
else:
    generations = [json.loads(line) for line in (REPLAY_SOURCE/'generations.jsonl').read_text().splitlines()]
    rows = saved['rows']
    (OUTPUT/'generations.jsonl').write_text((REPLAY_SOURCE/'generations.jsonl').read_text())


64-calibration-plain-00: 64 new tokens in 6.445s


64-calibration-plain-01: 64 new tokens in 5.534s


64-calibration-plain-02: 64 new tokens in 5.468s


64-calibration-plain-03: 64 new tokens in 5.519s


64-calibration-plain-04: 64 new tokens in 5.510s


64-calibration-plain-05: 64 new tokens in 5.472s


... (86 outputs left out)

128-evaluation-plain-10: 128 new tokens in 11.005s


128-evaluation-watermarked-10: 128 new tokens in 11.051s


128-evaluation-plain-11: 128 new tokens in 11.032s


128-evaluation-watermarked-11: 128 new tokens in 11.087s


In [ ]:
def audit_measurements(generations, rows, vocab_size, eos):
    assert len(generations) == 96 and len(rows) == 768
    by_id = {g['source_id']:g for g in generations}
    assert len(by_id) == 96
    combinations = collections.Counter((r['source_id'],r['condition'],r['key_condition']) for r in rows)
    assert len(combinations) == 768 and set(combinations.values()) == {1}
    for length in LENGTHS:
        for split, count, origins in [('calibration',24,['plain']),('evaluation',12,['plain','watermarked'])]:
            expected = sample_plan(length, split, count)
            for origin in origins:
                subset = sorted((g for g in generations if g['source_length']==length and
                                 g['split']==split and g['origin']==origin), key=lambda g:g['sample_index'])
                assert len(subset) == count
                for g, plan in zip(subset, expected):
                    assert all(g[k] == v for k,v in plan.items())
                    assert len(g['token_ids']) == length and g['prompt_token_ids'] and g['text']
                    assert g['source_id'] == f'{length}-{split}-{origin}-{plan["sample_index"]:02d}'
                    assert g['model_id'] == MODEL_ID and g['model_revision'] == REVISION
                    assert g['watermark_keys'] == KEYS
                    assert math.isfinite(g['generation_seconds']) and g['generation_seconds'] >= 0
                    if MODE == 'generate':
                        assert tokenizer(g['prompt'])['input_ids'] == g['prompt_token_ids']
                        assert tokenizer.decode(g['token_ids'], clean_up_tokenization_spaces=False) == g['text']
    cal = [g for g in generations if g['split']=='calibration']
    ev = [g for g in generations if g['split']=='evaluation']
    assert not ({g['prompt'] for g in cal} & {g['prompt'] for g in ev})
    assert not ({g['seed'] for g in cal} & {g['seed'] for g in ev})
    for r in rows:
        g = by_id[r['source_id']]
        assert all(r[k] == g[k] for k in ['source_length','split','origin','prompt','seed','sample_index','generation_seconds'])
        assert r['perturbation_seed'] == g['seed']+900000
        edited, edits = perturb(g['token_ids'], r['condition'], r['perturbation_seed'], vocab_size)
        assert r['token_ids'] == edited and r['edited_original_positions'] == edits
        score = real_score(edited, processors[r['key_condition']], eos)
        assert all(r[k] == v for k,v in score.items()), 'Original-ID score/mask/g-value mismatch'
        assert r['effective_tokens'] > 0 and math.isfinite(r['score']), 'No finite effective evidence'
        assert r['actual_token_count'] == len(edited)
        assert len(r['valid_mask']) == len(edited)-4
        assert len(r['g_values']) == len(r['valid_mask'])
        assert r['effective_tokens'] == sum(r['valid_mask'])
        rt = real_score(r['retokenized_token_ids'], processors[r['key_condition']], eos)
        assert r['retokenized_score'] == rt['score'] and r['retokenized_effective_tokens'] == rt['effective_tokens']
        assert rt['score'] is None or math.isfinite(rt['score'])
        assert r['roundtrip_equal'] == (edited == r['retokenized_token_ids'])
        assert 'retokenized_detected' not in r, 'Old thresholds must not classify retokenized diagnostics'
        if MODE == 'generate':
            assert tokenizer.decode(edited, clean_up_tokenization_spaces=False) == r['text']
            assert tokenizer.encode(r['text'], add_special_tokens=False) == r['retokenized_token_ids']
    summary, thresholds = summarize(rows)
    assert len(summary) == len(thresholds) == 16
    assert all(math.isfinite(t) for t in thresholds.values())
    assert all(s['calibration_count']==24 and s['plain_count']==s['watermarked_count']==12 for s in summary)
    assert all(s['calibration_detected']==0 for s in summary)
    for r in rows:
        if 'threshold' in r:
            assert r['threshold'] == thresholds[group_key(r)]
            assert r['detected'] == (r['score'] is not None and r['score'] > r['threshold'])
    return summary, thresholds

summary, thresholds = audit_measurements(generations, rows, VOCAB_SIZE, EOS)
for row in rows:
    row['threshold'] = thresholds[group_key(row)]
    row['detected'] = row['score'] is not None and row['score'] > row['threshold']
roundtrip = []
for origin in ['plain','watermarked']:
    for condition in CONDITIONS:
        subset = [r for r in rows if r['origin']==origin and r['condition']==condition and r['key_condition']=='correct']
        roundtrip.append(dict(origin=origin, condition=condition, count=len(subset),
                              mismatches=sum(not r['roundtrip_equal'] for r in subset)))
timing = (dict(model_load_seconds=load_seconds, installation_seconds=INSTALL_SECONDS,
    generation_and_scoring_seconds=time.perf_counter()-generation_start,
    sum_generate_seconds=sum(g['generation_seconds'] for g in generations),
    total_seconds=time.perf_counter()-NOTEBOOK_STARTED,
    peak_cuda_allocated_bytes=torch.cuda.max_memory_allocated(),
    peak_cuda_reserved_bytes=torch.cuda.max_memory_reserved()) if MODE=='generate' else saved['timing'])
result = dict(kind='real', parameters=params, environment=metadata, timing=timing,
    counts=dict(generations=len(generations), score_rows=len(rows),
        generation_groups=dict(collections.Counter(f'{g["source_length"]}:{g["split"]}:{g["origin"]}' for g in generations))),
    thresholds=thresholds, summary=summary, roundtrip=roundtrip, rows=rows)
if MODE=='replay':
    assert summary == saved['summary'] and thresholds == saved['thresholds'] and roundtrip == saved['roundtrip']
    result['replay'] = dict(mode='CPU saved-result scoring only; no fresh generation',
                           original_gpu_timing_preserved=True)
save_json(OUTPUT/'real.json', result)
audit_report = dict(generations_verified=96, score_rows_recomputed=768, thresholds_recomputed=16,
    prompt_seed_leakage=False, masks_and_g_values_verified=True, nonempty_evidence=True,
    text_and_prompt_tokenization_verified=(MODE=='generate'), mode=MODE)
save_json(OUTPUT/'audit.json', audit_report)
print(json.dumps({'counts':result['counts'], 'audit':audit_report}, indent=2))


{
  "counts": {
    "generations": 96,
    "score_rows": 768,
    "generation_groups": {
      "64:calibration:plain": 24,
      "64:evaluation:plain": 12,
      "64:evaluation:watermarked": 12,
      "128:calibration:plain": 24,
      "128:evaluation:plain": 12,
      "128:evaluation:watermarked": 12
    }
  },
  "audit": {
    "generations_verified": 96,
    "score_rows_recomputed": 768,
    "thresholds_recomputed": 16,
    "prompt_seed_leakage": false,
    "masks_and_g_values_verified": true,
    "nonempty_evidence": true,
    "text_and_prompt_tokenization_verified": true,
    "mode": "generate"
  }
}


In [ ]:
# Chart only reads saved JSON. No inference or download in this cell.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
chart_data = json.loads((OUTPUT/'real.json').read_text())
plt.rcParams.update({'figure.facecolor':'#1a1a2e','axes.facecolor':'#1a1a2e',
    'text.color':'#e0e0e0','axes.labelcolor':'#e0e0e0','axes.edgecolor':'#555',
    'xtick.color':'#e0e0e0','ytick.color':'#e0e0e0','grid.color':'#444','font.size':10})
fig, axes = plt.subplots(2,2,figsize=(15,10),sharey=True)
for ax, (length,key_name) in zip(axes.flat, [(n,k) for n in LENGTHS for k in ['correct','wrong']]):
    subset = [next(s for s in chart_data['summary'] if s['source_length']==length and
                   s['condition']==condition and s['key_condition']==key_name) for condition in CONDITIONS]
    for offset, origin, color, label in [(-0.19,'watermarked','#65d6ad','Watermarked detection'),
                                         (0.19,'plain','#ff9f69','Plain false positives')]:
        heights = [s[f'{origin}_detected']/s[f'{origin}_count'] for s in subset]
        bars = ax.bar([i+offset for i in range(4)], heights, width=.36, color=color, label=label)
        for bar,s in zip(bars,subset):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.025,
                    f'{s[f"{origin}_detected"]}/{s[f"{origin}_count"]}', ha='center',fontsize=9)
    ax.set_title(f'{length} source tokens / {key_name} key')
    ax.set_xticks(range(4),['Clean','Half prefix','Delete 10%','Replace 10%'])
    ax.set_ylim(0,1.16); ax.set_ylabel('Detected / 12 evaluation generations'); ax.grid(axis='y',alpha=.3)
axes[0,0].legend(loc='upper center',fontsize=8)
fig.suptitle('Qwen3-4B SynthID: detection and held-out false positives',fontsize=17)
fig.text(.5,.015,'96 real generations; paired token-ID edits; 24 plain calibration samples per threshold; mean g-values, not Bayesian probability.\nRaw English continuation (no chat/think template); independent thresholds per length/edit/key; re-encoding is diagnostic only.',ha='center',fontsize=9)
fig.tight_layout(rect=(0,.06,1,.95)); fig.savefig(OUTPUT/'summary.png',dpi=150,facecolor='#1a1a2e'); plt.close(fig)


In [ ]:
# Bundle only completed evidence; never include environment variables, credentials or model cache.
provenance = dict(model_id=MODEL_ID, model_revision=REVISION, mode=MODE,
    created_utc=datetime.now(timezone.utc).isoformat(),
    source_reference_sha256={'real_lab.py': '0b7a654f600cbf427c7b6f700be11776762faf9c11af11620a6168133f9e02fd', 'lab.py': '7cd30f2a6449ec5de1a640ae0cd99bf051c1210229ac85271580d6c54e1c76d1', 'audit.py': 'c5fac9cf2d89c66aee020881aad4864cea802da38d067cbc1835646b8891f716', 'main.py': '547d40cbfbe25e74125b440346786680a48d2718af227993365171c6fe60861b'},
    canonical_sampling_table_sha256=TABLE_SHA256,
    native_notebook='qwen3_gpu.ipynb', implementation='Transformers 4.57.1 official processor; CPU-seeded table copied to CUDA',
    raw_continuation=True, chat_template_used=False, think_prefix_used=False,
    audit=audit_report, artifacts={})
for name in ['generations.jsonl','real.json','summary.png','audit.json']:
    path=OUTPUT/name
    assert path.is_file() and path.stat().st_size > 0
    provenance['artifacts'][name] = dict(bytes=path.stat().st_size, sha256=hashlib.sha256(path.read_bytes()).hexdigest())
save_json(OUTPUT/'provenance.json', provenance)
with zipfile.ZipFile(OUTPUT/'results.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['generations.jsonl','real.json','summary.png','provenance.json','audit.json']:
        archive.write(OUTPUT/name,arcname=name)
with zipfile.ZipFile(OUTPUT/'results.zip') as archive:
    assert archive.testzip() is None
    for name in archive.namelist():
        assert archive.read(name) == (OUTPUT/name).read_bytes()
print(json.dumps({'output_directory':str(OUTPUT),'bundle_bytes':(OUTPUT/'results.zip').stat().st_size,
                   'generations':96,'score_rows':768,'threshold_groups':16,'mode':MODE},indent=2))


{
  "output_directory": "/content/synthid-qwen-output",
  "bundle_bytes": 570116,
  "generations": 96,
  "score_rows": 768,
  "threshold_groups": 16,
  "mode": "generate"
}


## Interpretation and audit limits
The saved masks/g-value vectors are the evidence for every score. Calibration is the maximum of 24 plain scores + 1e-12 per length/edit/key group, not a guaranteed production false-positive rate or a single unknown-edit deployment threshold. The 768 rows are paired derivatives of 96 outputs, not 768 independent generations. Re-encoded scores are diagnostics only: no original-ID threshold is applied to them.

`real.json` preserves the original chart-friendly schema. The historical `audit.py` hard-codes GPT-2 vocabulary/EOS and requires toy.json, so **do not run it unchanged on Qwen**; this notebook's native audit uses recorded Qwen vocabulary/EOS and independently re-scores every row. CPU replay does not prove GPU generation worked anew. A clean GPU Run all and actual output review remain necessary; offline tests establish structure and helper equivalence only.

Primary implementation references: Transformers v4.57.1 generation/configuration_utils.py and logits_process.py; the pinned Qwen model card. No claim of bitwise equality across devices, torch versions, or hardware. The actual torch/CUDA build is recorded, not assumed to be Colab's expected torch 2.11 / CUDA 13.
